# Aula de Revisão Integrada (Conteúdos das Aulas 01 a 07)

---

## Storytelling: Operação Auditoria Express — *Data Organized*

### O Contexto
Você acaba de completar o primeiro ciclo de imersão como estagiário(a) de dados na **Data Organized** (Curitiba-PR). Na manhã desta sexta-feira, o **Analista Sênior** convocou uma reunião de emergência: o diretor comercial apresentará os resultados de 2.000 contratos fechados em 2026 para o conselho de investidores, mas há suspeitas de que **valores atípicos (outliers)** estejam distorcendo o ticket médio e escondendo gargalos operacionais nas regiões.

### Sua Missão
Antes que os dados sejam liberados para o servidor de produção e para o dashboard executivo, você recebeu o arquivo `data_organized_contratos_v2000.csv` no seu *container* de homologação e precisa conduzir uma **auditoria completa em 6 etapas**, resgatando todas as ferramentas aprendidas nas **Aulas 01 a 07**:
* **Etapa 1 (Aula 01):** Diagnóstico estrutural, classificação de variáveis e tratamento de nulos com Pandas.
* **Etapa 2 (Aula 02):** Fatiamento (`loc`, `iloc`, `query`) e análise de quartis ($Q1, Q2, Q3$).
* **Etapa 3 (Aulas 03 e 04):** Simulação de consulta SQL (DQL) e cruzamento relacional (`merge` / `JOIN`) com a tabela de metas regionais.
* **Etapa 4 (Aula 05):** Conversão para `NumPy Array` e teste de confiabilidade da média (Distância Média-Mediana).
* **Etapa 5 (Aula 06):** Detecção matemática de *Outliers* via Intervalo Interquartil ($IQR$) e ordenação (`sort_values`).
* **Etapa 6 (Aula 07):** Cálculo de amplitude total e construção de painel visual 1x2 (Boxplot + Resumo Estatístico).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# Caminho dinâmico para a pasta 'dados' e o arquivo 'tabela.csv'
caminho = Path('DATA') / 'data_organized_contratos_v2000.csv'

# 1. Carregar base de dados
df = pd.read_csv(caminho, encoding='latin1')

df.head()

,ID_Contrato,mes,Regiao_Cliente,Linha_Produto,Qtd_Licencas,Valor_Contrato_BRL,Dias_Fechamento
0,TRX-10000,06-Jun,Sudeste,CiberseguranÃ§a,1,2605.80,11.0
1,TRX-10001,12-Dez,Sul,LicenÃ§as SaaS,1,2030.32,27.0
2,TRX-10002,10-Out,Nordeste,LicenÃ§as SaaS,19,14988.39,20.0
3,TRX-10003,08-Ago,Sul,Hardware & Servidores,5,3408.11,9.0
4,TRX-10004,03-Mar,Sudeste,Hardware & Servidores,13,5277.71,NaN


---
## Etapa 1 (Revisão Aula 01): Anatomia da Base, Variáveis e Dados Nulos

### Perguntas Norteadoras
1. Quais colunas da base são **variáveis qualitativas** e quais são **variáveis quantitativas** (discretas ou contínuas)?
2. Existe alguma coluna com valores ausentes (`NaN`)? Como uma `Series` do Pandas lida com valores nulos ao calcular a média em comparação com uma lista comum do Python?
3. Qual é o faturamento total (`sum`), o ticket médio (`mean`), o menor (`min`) e o maior (`max`) contrato registrado?

In [4]:
# Desenvolva aqui o código da Etapa 1 (df.info(), isna().sum(), sum(), mean(), min(), max()):
import pandas as pd

# Carregando o arquivo CSV de vendas
arquivo = Path("..") / "Data" / "dados_organized_contratos_v2000.csv"
df_vendas = pd.read_csv(arquivo)
df_vendas.info()

NameError: name 'Path' is not defined

---
## Etapa 2 (Revisão Aula 02): Fatiamento Estratégico e Quartis ($Q1, Q2, Q3$)

### Perguntas Norteadoras
1. Utilizando `.query()`, filtre apenas os contratos da região **'Sudeste'** na linha **'Cloud & Storage'**. Quantos contratos atendem a esse critério e qual o valor total gerado por esse nicho?
2. Utilizando `.iloc`, exiba apenas as linhas de índice `10` a `15` e as colunas `0`, `2` e `4`.
3. Calcule os quartis **$Q1$ (25%)**, **$Q2$ (50% - Mediana)** e **$Q3$ (75%)** da coluna `Valor_Contrato_BRL`. O que o valor de $Q3$ informa sobre 75% dos contratos fechados pela empresa?

In [ ]:
# Desenvolva aqui o código da Etapa 2 (query, iloc, describe() e np.percentile):



---
## Etapa 3 (Revisão Aulas 03 e 04): Visão Relacional (SQL DQL + Cruzamento de Fontes)

O setor financeiro enviou uma planilha auxiliar com a **Meta de Ticket Médio** e o **Gerente Responsável** por cada região (simulando uma tabela `dim_regioes` vinda do MySQL).

### Perguntas Norteadoras
1. Como você escreveria a consulta **SQL (DQL)** equivalente para obter o total de vendas agrupado por `Regiao_Cliente` ordenado do maior para o menor?
2. Utilizando `pd.merge()` (equivalente ao `INNER JOIN` / `LEFT JOIN` do SQL), integre o resumo regional do DataFrame `df` com a tabela `df_gerentes` abaixo e identifique quais regiões superaram a meta de ticket médio.

In [3]:
# Tabela auxiliar simulando extração do banco MySQL (dim_regioes)
df_gerentes = pd.DataFrame({
    'Regiao_Cliente': ['Sul', 'Sudeste', 'Centro-Oeste', 'Nordeste', 'Norte'],
    'Gerente_Regional': ['Carlos Mendes', 'Fernanda Lima', 'Roberto Alves', 'Juliana Costa', 'Marcos Souza'],
    'Meta_Ticket_BRL': [4800.00, 5200.00, 4500.00, 4700.00, 4600.00]
})

# Desenvolva aqui o agrupamento por região e o cruzamento via pd.merge():



---
## Etapa 4 (Revisão Aula 05): NumPy Arrays e Confiabilidade da Média

### Perguntas Norteadoras
1. Converta a coluna `Valor_Contrato_BRL` em um **Array NumPy** e calcule a média (`np.mean`) e a mediana (`np.median`).
2. Calcule a distância relativa entre média e mediana pela fórmula do PTD: $\text{Distância} = \frac{\text{Média} - \text{Mediana}}{\text{Mediana}}$.
3. Sabendo que uma distância $\ge |25\%|$ indica **alta influência de valores extremos**, a média é uma medida confiável para o diretor apresentar ao conselho? A distorção é puxada por valores maiores ou menores?

In [ ]:
# Desenvolva aqui o código da Etapa 4 (np.array, mean, median e regra de distância):



---
## Etapa 5 (Revisão Aula 06): Caça aos Outliers via IQR

### Perguntas Norteadoras
1. Calcule o **Intervalo Interquartil ($IQR = Q3 - Q1$)**, o **Limite Inferior ($Q1 - 1,5 \times IQR$)** e o **Limite Superior ($Q3 + 1,5 \times IQR$)** para `Valor_Contrato_BRL`.
2. Quantos contratos são **Outliers Superiores** e quantos são **Outliers Inferiores**?
3. Exiba os **5 maiores outliers superiores** ordenados de forma decrescente (`sort_values(ascending=False)`). A quais regiões e linhas de produto eles pertencem?

In [ ]:
# Desenvolva aqui o código da Etapa 5 (IQR, limites e filtragem/ordenação de outliers):



---
## Etapa 6 (Revisão Aula 07): Amplitude Total e Painel Executivo (1x2)

### Perguntas Norteadoras
1. Qual é a **Amplitude Total** (`max - min`) dos valores de contrato?
2. Construa um painel no Matplotlib com **1 linha e 2 colunas (`plt.subplots(1, 2)`)**, exibindo na esquerda o **Boxplot** de `Valor_Contrato_BRL` e na direita um **cartão de resumo textual (`plt.text`)** com todas as medidas calculadas (Mínimo, Limite Inferior, Q1, Mediana, Média, Q3, Limite Superior, Máximo, IQR e Amplitude).
3. **Veredito Final para o Analista Sênior:** Que recomendação você dá ao diretor comercial antes da reunião com os investidores?

In [ ]:
# Desenvolva aqui o código da Etapa 6 (Painel 1x2 com Boxplot + plt.text):

